In [ ]:
# Step 1: fetch and parse the Berlin page

import requests
from bs4 import BeautifulSoup

url = "https://en.wikipedia.org/wiki/Berlin"
headers = {"User-Agent": "Mozilla/5.0 (educational project)"}

response = requests.get(url, headers=headers)
print(response.status_code)  # should be 200

soup = BeautifulSoup(response.text, "lxml")

200


In [ ]:
# Step 1: fetch and parse the Hamburg page

url = "https://en.wikipedia.org/wiki/Hamburg"
headers = {"User-Agent": "Mozilla/5.0 (educational project)"}

response = requests.get(url, headers=headers)
print(response.status_code)  # should be 200

soup = BeautifulSoup(response.text, "lxml")

200


In [ ]:
# Step 1: fetch and parse the Munich page

url = "https://en.wikipedia.org/wiki/Munich"
headers = {"User-Agent": "Mozilla/5.0 (educational project)"}

response = requests.get(url, headers=headers)
print(response.status_code)  # should be 200

soup = BeautifulSoup(response.text, "lxml")

200


In [17]:
# Step 2: find the raw population and coordinates

infobox = soup.find("table", class_="infobox")

# Print every row that mentions "Population", plus the rows right after it
rows = infobox.find_all("tr")
for i, row in enumerate(rows):
    header = row.find("th")
    if header and "Population" in header.get_text():
        for r in rows[i : i + 4]:
            print(repr(r.get_text(" ", strip=True)))
        break

'Population (2024-12-31) [ 2 ]'
'• City 1,973,896'
'• Rank 2nd in Germany'
'• Density 2,614.1/km 2 (6,770.5/sq mi)'


In [18]:
population_raw = rows[i + 1].find("td").get_text(strip=True)
print(population_raw)  # raw and messy for now, cleaning comes later

1,973,896


In [19]:
lat_raw = soup.find("span", class_="latitude").get_text()
lon_raw = soup.find("span", class_="longitude").get_text()
print(lat_raw, lon_raw)  # something like 52°31′12″N

53°33′N 10°00′E


In [20]:
import re

def clean_population(raw):
    """'3,596,999' -> 3596999 (also drops footnote markers like [1])"""
    raw = raw.split("[")[0]
    return int(re.sub(r"\D", "", raw))

def dms_to_decimal(dms):
    """'52°31′12″N' -> 52.52 (S and W become negative)"""
    deg, minutes, seconds, direction = re.match(
        r"(\d+)°(?:(\d+)′)?(?:(\d+)″)?([NSEW])", dms
    ).groups()
    value = int(deg) + int(minutes or 0) / 60 + int(seconds or 0) / 3600
    return round(-value if direction in ("S", "W") else value, 4)

print(clean_population(population_raw))  # 3596999
print(dms_to_decimal(lat_raw), dms_to_decimal(lon_raw))  # ~52.52, ~13.405

1973896
53.55 10.0


In [ ]:
# next step: the loop and DataFrame - full code in one cell

import re
import time

import pandas as pd
import requests
from bs4 import BeautifulSoup

HEADERS = {"User-Agent": "Mozilla/5.0 (educational project)"}


def clean_population(raw):
    """'3,596,999' -> 3596999 (drops footnote markers like [1])"""
    raw = raw.split("[")[0]
    return int(re.sub(r"\D", "", raw))


def dms_to_decimal(dms):
    """'52°31′12″N' -> 52.52 (S and W become negative)"""
    deg, minutes, seconds, direction = re.match(
        r"(\d+)°(?:(\d+)′)?(?:(\d+)″)?([NSEW])", dms
    ).groups()
    value = int(deg) + int(minutes or 0) / 60 + int(seconds or 0) / 3600
    return round(-value if direction in ("S", "W") else value, 4)


def get_infobox_value(infobox, label, next_row=False):
    """Return the text of the row whose header contains `label`.
    With next_row=True, return the value from the row after the header
    (needed for Population, where the number sits one row below)."""
    rows = infobox.find_all("tr")
    for i, row in enumerate(rows):
        header = row.find("th")
        if header and label in header.get_text():
            target = rows[i + 1] if next_row else row
            return target.find("td").get_text(" ", strip=True)
    return None


def scrape_city(city):
    url = f"https://en.wikipedia.org/wiki/{city}"
    response = requests.get(url, headers=HEADERS)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "lxml")
    infobox = soup.find("table", class_="infobox")

    return {
        "city": city,
        "country": get_infobox_value(infobox, "Country"),
        "population": clean_population(
            get_infobox_value(infobox, "Population", next_row=True)
        ),
        "latitude": dms_to_decimal(soup.find("span", class_="latitude").get_text()),
        "longitude": dms_to_decimal(soup.find("span", class_="longitude").get_text()),
    }


cities = ["Berlin", "Hamburg", "Munich"]

rows_list = []
for city in cities:
    rows_list.append(scrape_city(city))
    time.sleep(1)  # be polite to Wikipedia

df = pd.DataFrame(rows_list)
print(df.dtypes)
df

city              str
country           str
population      int64
latitude      float64
longitude     float64
dtype: object


,city,country,population,latitude,longitude
0,Berlin,Germany,3596999,52.5200,13.405
1,Hamburg,Germany,1973896,53.5500,10.000
2,Munich,Germany,1505036,48.1375,11.575


In [23]:
# replacing with importing the get_cities_data function from src.scraping

import sys
sys.path.append("..")  # lets the notebook see the src/ folder

from src.scraping import get_cities_data

df = get_cities_data(["Berlin", "Hamburg", "Munich"])
df

,city,country,population,latitude,longitude
0,Berlin,Germany,3596999,52.5200,13.405
1,Hamburg,Germany,1973896,53.5500,10.000
2,Munich,Germany,1505036,48.1375,11.575
